In [1]:
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from openai import OpenAI

In [2]:
load_dotenv()
openai_client=OpenAI()
model=SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
from ingest import load_faq_data, build_index
documents=load_faq_data()
index=build_index(documents)

In [4]:
from rag_helper import RAGBase

assistant=RAGBase(
index=index,
llm_client=openai_client,
)

In [5]:
query='Can I still join the course after the registration window?'
assistant.rag(query)

'Yes, you can still join the course after the registration window, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

In [7]:
texts=[]

for doc in documents:
    text=doc['question']+' '+doc['answer']
    texts.append(text)

from tqdm import tqdm
batch=100

vectors=[]

for i in tqdm(range(0, len(texts), batch)):
    batch_texts=texts[i:i+batch]
    embeddings=model.encode(batch_texts)
    vectors.extend(embeddings)

100%|██████████| 14/14 [01:32<00:00,  6.61s/it]


In [ ]:
from minsearch import VectorSearch

vindex=VectorSearch(keyword_fields=['course'])
vindex.fit(vectors, documents)

### Vector RAG Implementation

In [6]:
class RAGVector(RAGBase):
    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder=embedder

    def search(self, query, num_results=5):
        query_vector=self.embedder.encode(query)
        filter_dict={'course':self.course}

        return self.index.search(query_vector, 
                                 num_results=num_results, 
                                 filter_dict=filter_dict)
    

In [11]:
vector_assistant=RAGVector(
embedder=model,
index=vindex,
llm_client=openai_client)

In [13]:
vector_assistant.rag(query)

'Yes, you can still join the course after the registration window, but if you want to receive a certificate, you must submit your project while submissions are still being accepted.'

In [14]:
print('check')

check


### Using sqlitesearch

In [16]:
from sqlitesearch import VectorSearchIndex

vs_index=VectorSearchIndex(keyword_fields=['course'], 
                           mode='ivf',
                           db_path='faq_vectors2.db')

In [17]:
vs_index.fit(vectors, documents)

In [34]:
query

'Can I still join the course after the registration window?'

In [21]:
query_vector=model.encode(query)

In [24]:
vs_index.search(query_vector, num_results=5)

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '73ed660aa5',
  'course': 'ai-dev-tools-zoomcamp',
  'se

In [25]:
assistant=RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client
)

In [32]:
assistant.rag('When do the cohort start?')

"I don't know."

In [33]:
vs_index.close()